# 🔬 Shotgun Sequencing Simulation — STAT 623, Rice University

**▶ Run this notebook top-to-bottom.** It loads all helper notebooks via `%run`, runs both simulations, and saves 9 PNG output figures.

---

| Notebook loaded | Contents |
|---|---|
| `theoretical_stats.ipynb` | α, E[contigs], E[contig length] |
| `genome_utils.ipynb` | Genome generation + read sampling |
| `overlap.ipynb` | Overlap graph |
| `alignment_visualization.ipynb` | Alignment, contigs, coverage, plots |
| `summary_plots.ipynb` | 6-panel summary figure |
| `extended_plots.ipynb` | 5 statistical plots + α sweep |


## 1 · Install dependencies


In [ ]:
# Uncomment if running on Google Colab
# !pip install networkx matplotlib numpy


## 2 · Load all modules

`%run` executes each notebook and makes its functions available here.


In [ ]:
%run theoretical_stats.ipynb
%run genome_utils.ipynb
%run overlap.ipynb
%run alignment_visualization.ipynb
%run summary_plots.ipynb
%run extended_plots.ipynb


## 3 · Additional imports


In [ ]:
import random
import math
import os

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.collections import LineCollection
import networkx as nx


## 4 · Simulation helpers


#### `_collect_stats`
*Align reads and collect all metrics needed for plots and printing.*


In [ ]:
def _collect_stats(genome: str, reads: list) -> dict:
    """Align reads and collect all metrics needed for plots and printing.

    Returns a dict with keys:
        reads_data, contigs, coverage, N, G, L, alpha,
        exp_contigs, exp_contig_len, mapped_reads,
        contig_lengths, read_lengths
    """
    reads_data  = create_alignment(genome, reads)
    contigs     = get_contigs(reads_data)
    coverage    = get_coverage(len(genome), reads_data)

    N     = len(reads)
    G     = len(genome)
    L     = max(len(r) for r in reads) if reads else 0
    alpha = compute_alpha(N, L, G)

    return {
        "reads_data":      reads_data,
        "contigs":         contigs,
        "coverage":        coverage,
        "N":               N,
        "G":               G,
        "L":               L,
        "alpha":           alpha,
        "exp_contigs":     expected_contigs(N, alpha),
        "exp_contig_len":  expected_contig_length(G, N, alpha),
        "mapped_reads":    len(reads_data),
        "contig_lengths":  [e - s for s, e in contigs],
        "read_lengths":    [len(r) for r in reads],
    }


#### `_print_stats`
*Pretty-print simulation statistics.*


In [ ]:
def _print_stats(label: str, s: dict) -> None:
    """Pretty-print simulation statistics."""
    print(f"\n{'='*54}")
    print(f"  {label}")
    print(f"{'='*54}")
    print(f"  Genome length    : {s['G']} bp")
    print(f"  Number of reads  : {s['N']}")
    print(f"  Mapped reads     : {s['mapped_reads']}  "
          f"({100*s['mapped_reads']/s['N']:.1f}%)")
    print(f"  Number of contigs: {len(s['contigs'])}")
    if s["contig_lengths"]:
        print(f"  Max contig length: {max(s['contig_lengths'])} bp")
        print(f"  Avg contig length: {sum(s['contig_lengths'])/len(s['contig_lengths']):.1f} bp")
    print(f"  Max coverage     : {max(s['coverage'])}")
    print(f"  Mean coverage    : {sum(s['coverage'])/len(s['coverage']):.2f}x")
    print(f"  [Theory] α       = {s['alpha']:.4f}")
    print(f"  [Theory] Expected contigs      ≈ {s['exp_contigs']:.2f}")
    print(f"  [Theory] Expected contig len   ≈ {s['exp_contig_len']:.2f} bp")


## 5 · Simulation — random genome (no repeats)

Lander-Waterman Poisson model holds for a purely random genome.


#### `shotgun_simulation_without_repeats`
*Simulate shotgun sequencing on a purely random genome.*


In [ ]:
def shotgun_simulation_without_repeats() -> dict:
    """Simulate shotgun sequencing on a purely random genome.

    Returns the full stats dict for use in summary plots.
    """
    genome = generate_genome(2000)
    reads  = generate_reads(genome, num_reads=1200, min_len=50, max_len=100)

    # Overlap graph
    graph = build_overlap_graph(reads, min_length=5)
    draw_graph(graph, "overlap_structure_graph_without_repeats")

    # Alignment & coverage plot
    visualize_alignment(genome, reads, "alignment_plot_without_repeats.png")

    # Statistics
    s = _collect_stats(genome, reads)
    _print_stats("Genome WITHOUT repeats", s)

    # Contig-only plot
    fig, ax = plt.subplots(figsize=(12, 4))
    plot_contigs(ax, s["contigs"], show_labels=True, show_gaps=True)
    ax.set_xlim(0, len(genome))
    ax.set_title("Contig Assembly Results (Without Repeats)")
    ax.set_xlabel("Genome Position")
    ax.set_yticks([])
    plt.tight_layout()
    plt.savefig("contigs_only_without_repeats.png", dpi=150)
    plt.close(fig)
    print("Contig plot saved to 'contigs_only_without_repeats.png'")

    s["genome"] = genome          # store for summary plots
    return s


## 6 · Simulation — repeat-rich genome

Repeat elements longer than L violate the Poisson assumption.


#### `shotgun_simulation_with_repeats`
*Simulate shotgun sequencing on a repeat-rich genome.*


In [ ]:
def shotgun_simulation_with_repeats() -> dict:
    """Simulate shotgun sequencing on a repeat-rich genome.

    Returns the full stats dict for use in summary plots.
    """
    genome = generate_genome_with_repeats(2000)
    reads  = generate_reads(genome, num_reads=1200, min_len=50, max_len=100)

    # Overlap graph
    graph = build_overlap_graph(reads, min_length=5)
    draw_graph(graph, "overlap_structure_graph_with_repeats")

    # Alignment & coverage plot
    visualize_alignment(genome, reads, "alignment_plot_with_repeats.png",
                        repeat_target="CAG" * 5)

    # Statistics
    s = _collect_stats(genome, reads)
    _print_stats("Genome WITH repeats", s)

    # Contig-only plot
    fig, ax = plt.subplots(figsize=(12, 4))
    plot_contigs(ax, s["contigs"], show_labels=True, show_gaps=True)
    ax.set_xlim(0, len(genome))
    ax.set_title("Contig Assembly Results (With Repeats)")
    ax.set_xlabel("Genome Position")
    ax.set_yticks([])
    plt.tight_layout()
    plt.savefig("contigs_only_with_repeats.png", dpi=150)
    plt.close(fig)
    print("Contig plot saved to 'contigs_only_with_repeats.png'")

    s["genome"] = genome
    return s


# =============================================================================
# SUMMARY & COMPARISON PLOTS
# =============================================================================


## 7 · Main orchestrator


#### `main`
*Run both simulations, then produce all summary and extended plots.*


In [ ]:
def main():
    """Run both simulations, then produce all summary and extended plots."""
    s0 = shotgun_simulation_without_repeats()
    s1 = shotgun_simulation_with_repeats()

    # 6-panel overview comparison
    generate_summary_plots(s0, s1, output_png="summary_comparison.png")

    # Detailed Lander-Waterman α-sweep (standalone, high-resolution)
    plot_lander_waterman_sweep(
        G        = s0["G"],
        min_len  = min(s0["read_lengths"]),
        max_len  = max(s0["read_lengths"]),
        n_trials = 3,
        output_png = "lander_waterman_sweep.png",
    )

    # All 5 recommended extended plots in one figure
    generate_extended_summary_plots(
        s0,
        s1,
        output_png    = "extended_summary.png",
        sweep_trials  = 2,          # raise to 4-5 for smoother α-sweep dots
    )

    print("\n" + "="*54)
    print("  All output files")
    print("="*54)
    for fn in [
        "overlap_structure_graph_without_repeats.png",
        "overlap_structure_graph_with_repeats.png",
        "alignment_plot_without_repeats.png",
        "alignment_plot_with_repeats.png",
        "contigs_only_without_repeats.png",
        "contigs_only_with_repeats.png",
        "summary_comparison.png",
        "lander_waterman_sweep.png",
        "extended_summary.png",
    ]:
        print(f"  • {fn}")


if __name__ == "__main__":
    main()


## 8 · Run everything

Executes both simulations and saves **9 PNG files** to the current working directory.


In [ ]:
main()
